# Week 2 Day 2

## Covered Today
1. Building Data Science UIs with Gradio (No Front-End Skills Required)
2. Building Your First Gradio Interface with Callbacks and Sharing
3. Building Gradio Interfaces with Authentication and GPT Integration
4. Markdown Responses and Streaming with Gradio and OpenAI
5. Building Multi-Model Gradio UIs with GPT and Claude Streaming

In [1]:
import os
from dotenv import load_dotenv
import requests
from openai import OpenAI
from IPython.display import Markdown, display, update_display

### Now we load all the API Keys

In [2]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

### Now check if all loaded keys exist and are as per the format

In [3]:
if openai_api_key:
    if openai_api_key.startswith("sk-"):
        print(f"OpenAI      : OK           (begins {openai_api_key[:7]}...)")
    else:
        print("OpenAI      : WRONG FORMAT (should start with 'sk-')")
else:
    print("OpenAI      : MISSING")


if anthropic_api_key:
    if anthropic_api_key.startswith("sk-ant-"):
        print(f"Anthropic   : OK           (begins {anthropic_api_key[:10]}...)")
    else:
        print("Anthropic   : WRONG FORMAT (should start with 'sk-ant-')")
else:
    print("Anthropic   : MISSING")


if google_api_key:
    if google_api_key.startswith("AQ.Ab"):
        print(f"Google      : OK           (begins {google_api_key[:5]}...)")
    else:
        print("Google      : WRONG FORMAT (should start with 'AQ.Ab' or 'AIza')")
else:
    print("Google      : MISSING")


if openrouter_api_key:
    if openrouter_api_key.startswith("sk-or-"):
        print(f"OpenRouter  : OK           (begins {openrouter_api_key[:8]}...)")
    else:
        print("OpenRouter  : WRONG FORMAT (should start with 'sk-or-')")
else:
    print("OpenRouter  : MISSING")

OpenAI      : OK           (begins sk-proj...)
Anthropic   : OK           (begins sk-ant-api...)
Google      : OK           (begins AQ.Ab...)
OpenRouter  : OK           (begins sk-or-v1...)


In [4]:
# create clients for each provider
# for openai, we simply use OpenAI, for others we need to specify the base url and key, while using openai api library
openai_client = OpenAI()

google_url = 'https://generativelanguage.googleapis.com/v1beta/openai/'
anthropic_url = 'https://api.anthropic.com/v1/'
openrouter_url = 'https://openrouter.ai/api/v1'
ollama_url = 'http://127.0.0.1:11434/v1'

In [5]:
google_client = OpenAI(base_url=google_url, api_key=google_api_key)
anthropic_client = OpenAI(base_url=anthropic_url, api_key=anthropic_api_key)
openrouter_client = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama_client = OpenAI(base_url=ollama_url, api_key='Ollama')

Now, we start with Gradio. Gradio is a library that enables creation of frontend UIs using python code. For this, we start by importing Gradio. And the convention is to > import gradio as gr. We start by installing the lib in our venv by using uv add gradio. 

Gradio added, now we import it.

In [6]:
import gradio as gr

In [7]:
# next we start by writing a simple call gpt function in which we can enter a prompt
system_message = "You are a helpful assistant. Respond in simple text(non markdown)"

In [8]:
def call_gpt(prompt):
    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': prompt}
    ]
    response = openai_client.chat.completions.create(model='gpt-4.1-mini', messages=messages)
    return response.choices[0].message.content

In [9]:
call_gpt('How are you?')

"I'm doing well, thank you! How can I assist you today?"

Now we move to user interface

In [10]:
# we start with a simple function to understand how gradio works
def shout(text):
    print(f"Shout has been called with {text}")
    return text.upper()

In [11]:
shout("Hello")

Shout has been called with Hello


'HELLO'

In [12]:
gr.Interface(fn=shout, inputs='textbox', outputs='textbox', flagging_mode='never').launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [13]:
# next we try sharing gradio using the same last function
def shout(text):
    return text.upper()

In [14]:
gr.Interface(fn=shout, inputs='textbox', outputs='textbox', flagging_mode='never').launch(share=True)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://16778ae3f23848e585.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [15]:
# next we move to adding some more elements in the gradio interface rather than using the def version
message_input = gr.Textbox(label='Enter Message:', info="Enter a message to make it shouted", lines=7)
message_output = gr.Textbox(label='Response', lines=8)

view = gr.Interface(
    fn=shout,
    title='Shout',
    inputs=[message_input],
    outputs=[message_output],
    examples=['hello', 'howdy'],
    flagging_mode='never'
)
view.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [16]:
# next up, we take the same interface details and implement for GPT conversation
message_input = gr.Textbox(label='Enter Prompt', lines=8)
message_output = gr.Textbox(label='Response', lines=8)

view = gr.Interface(
    inputs=[message_input],
    outputs=[message_output],
    title='Gpt Bot',
    fn=call_gpt,
    flagging_mode='never',
    examples=['How are you?', 'Tell me about transformers in LLMs']
)

view.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [17]:
# next up, we make a tiny change, to show the output in markdown, for this, we make changes to system_message and rewrite the function (though we can do without it also, but just for practice)

system_message = 'You are a helpful assistant. You respond in markdown'

def call_gpt(prompt):
    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': prompt}
    ]
    response = openai_client.chat.completions.create(model='gpt-4.1-mini', messages=messages)

    return (response.choices[0].message.content)

In [18]:
# now we run the above once to check if all's working.
call_gpt('in one line tell me, what is a transformer in LLMs')

'A transformer in LLMs is a deep learning model architecture that uses self-attention mechanisms to process and generate sequences of data efficiently.'

In [19]:
# now, since the above was asked in one line, hence, no markdown output came, now, lets run it again to check, markdown is working.
call_gpt("Tell me what are transformers in LLMs")

'**Transformers** in Large Language Models (LLMs) refer to a specific type of neural network architecture that has revolutionized natural language processing (NLP). Introduced in the paper ["Attention is All You Need" (Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762), transformers are designed to handle sequential data (like text) efficiently and effectively.\n\n### Key Concepts of Transformers:\n\n1. **Self-Attention Mechanism**  \n   - Allows the model to weigh the importance of different words in a sentence relative to each other, regardless of their position.  \n   - For example, in the sentence "The cat sat on the mat," the model can directly relate "cat" and "sat" even though they are separated by other words.\n\n2. **Positional Encoding**  \n   - Since transformers do not process data sequentially like RNNs, they use positional encodings to give the model information about word order in a sentence.\n\n3. **Encoder-Decoder Structure** (in original transformer)  \n   - The

In [20]:
# awesome, markdown works fine, now we just take this function and implement gradio UI on top of this
input_message = gr.Textbox(label='Ask your question', lines=8)
output_message = gr.Markdown(label='Answer')

view = gr.Interface(
    fn=call_gpt,
    inputs=[input_message],
    outputs=[output_message],
    title='Your personal AI ChatBot',
    flagging_mode='never',
    examples=['What are transformers in LLMs?', 'What is life?']
)

view.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


As we see above, results are being published in markdown on the right, with proper formatting

In [21]:
# next on, we simply convert this to streaming
def stream_gpt(prompt):
    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': prompt}
    ]

    stream = openai_client.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )

    result = ""

    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result



In [22]:
input_message = gr.Textbox(label='Ask your question', lines=8)
output_message = gr.Markdown(label='Answer')

view = gr.Interface(
    fn=stream_gpt,
    inputs=[input_message],
    outputs=[output_message],
    title='Your personal AI ChatBot',
    flagging_mode='never',
    examples=['What are transformers in LLMs?', 'What is life?']
)

view.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


And now we were able to stream the results within gradio UI

In [23]:
# next up, we create a multi model chat engine, with the option of selecting LLM provider to ask questions
# for this, we first start by checking if APIs are working fine for anthropic, ollama and gemini
messages = [
    {'role': 'user', 'content': 'How are you?'}
]
response = anthropic_client.chat.completions.create(model='claude-haiku-4-5-20251001', messages=messages) #type: ignore
print(f"Anthropic Response: {response.choices[0].message.content}")

response = google_client.chat.completions.create(model='gemini-3.1-flash-lite', messages=messages) #type: ignore
print(f"Gemini Response: {response.choices[0].message.content}")

response = ollama_client.chat.completions.create(model='llama3.2:latest', messages=messages) #type: ignore
print(f"Ollama Response: {response.choices[0].message.content}")


Anthropic Response: I'm doing well, thanks for asking! I'm here and ready to help with whatever you need. How are you doing today?
Gemini Response: I'm doing great, thank you for asking! How are you doing today? Is there anything I can help you with?
Ollama Response: I'm just a language model, so I don't have emotions or feelings like humans do. However, I'm functioning properly and ready to assist you with any questions or tasks you may have! How can I help you today?


As we can see, all three APIs are working fine and responsive, we now move to creating streaming functions for each of these

In [24]:
def call_claude(prompt):
    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': prompt}
    ]

    stream = anthropic_client.chat.completions.create(
        model='claude-haiku-4-5-20251001',
        messages=messages,
        stream=True
    )

    response = ''
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [25]:
def call_gemini(prompt):
    messages = [
            {'role': 'system', 'content': system_message},
            {'role': 'user', 'content': prompt}
        ]
    
    stream = google_client.chat.completions.create(
            model='gemini-3.1-flash-lite',
            messages=messages,
            stream=True
        )
    
    response = ''
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [26]:
def call_ollama(prompt):
    messages = [
            {'role': 'system', 'content': system_message},
            {'role': 'user', 'content': prompt}
        ]
    
    stream = ollama_client.chat.completions.create(
            model='llama3.2:latest',
            messages=messages,
            stream=True
        )
    
    response = ''
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [27]:
# now, we create a single function to call any of these functions
def stream_model(prompt, model):
    if model == 'GPT':
        result = stream_gpt(prompt)
    elif model == 'Claude':
        result = call_claude(prompt)
    elif model == 'Gemini':
        result  = call_gemini(prompt)
    elif model == 'Ollama':
        result = call_ollama(prompt)
    else:
        raise ValueError("Unknown Model")
    yield from result #type: ignore

In [28]:
input_question = gr.Textbox(label="Question", lines=5)
model_selector = gr.Dropdown(['GPT', 'Claude', 'Gemini', 'Ollama'], label='Select Model', value='GPT')
output_section = gr.Markdown(label='Answer:')

view = gr.Interface(
    fn=stream_model,
    inputs=[input_question, model_selector],
    outputs=[output_section],
    title='Multi Model Conversation',
    flagging_mode='never',
    # examples=['Explain how LLMs work to a layman.', 'Explain "Attention is all you need" to a layman.']
)

view.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
